# Финальный проект по Process Mining

Ноутбук решает задание из `task.pdf` на датасете `final_dataset.csv`:

- строит event log и базовую аналитику процесса;
- кластеризует cases по процессным, категориальным и текстовым признакам;
- считает качество кластеризации через Silhouette и DBCV;
- строит DFG и сеть Петри через Alpha miner;
- считает process metrics по кластерам, частые циклы и feature importance для длительности процесса;
- делает кластеризацию и краткую суммаризацию текстовых признаков.

Перед запуском установите зависимости из корня этой папки:

```bash
pip install -r requirements.txt
```

В ноутбуке нет `pip install`, чтобы запуск оставался контролируемым и воспроизводимым.

In [ ]:
from pathlib import Path
import warnings
from collections import Counter

import numpy as np
import pandas as pd

from IPython.display import display, Markdown

from scipy import sparse
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

import hdbscan
from hdbscan.validity import validity_index

import pm4py
from pm4py.algo.discovery.alpha import algorithm as alpha_miner
from pm4py.algo.evaluation.replay_fitness import algorithm as replay_fitness_evaluator
from pm4py.visualization.petri_net import visualizer as pn_visualizer

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
DATA_PATH = Path("final_dataset.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

CASE_ID = "case concept:name"
ACTIVITY = "event concept:name"
TIMESTAMP = "event time:timestamp"
VALUE_COL = "event Cumulative net worth (EUR)"

REQUIRED_COLUMNS = [CASE_ID, ACTIVITY, TIMESTAMP]
TEXT_COLUMNS = [
    "case Spend area text",
    "case Sub spend area text",
    "case Spend classification text",
    "case Name",
]
CATEGORICAL_COLUMNS = [
    "case Spend area text",
    "case Company",
    "case Document Type",
    "case Sub spend area text",
    "case Purch. Doc. Category name",
    "case Vendor",
    "case Item Type",
    "case Item Category",
    "case Spend classification text",
    "case Source",
    "case GR-Based Inv. Verif.",
    "case Goods Receipt",
]

SILHOUETTE_SAMPLE_SIZE = 10_000
DBCV_SAMPLE_SIZE = 5_000
PM4PY_SAMPLE_CASES = 5_000
DFG_TOP_EDGES = 45

print("Рабочая директория:", Path.cwd())
print("Данные:", DATA_PATH.resolve())

## 1. Загрузка и базовая проверка данных

Ключевые поля для process mining:

- `case concept:name` — идентификатор case;
- `event concept:name` — activity/event;
- `event time:timestamp` — timestamp события.

Timestamp парсится с `dayfirst=True`, потому что в датасете формат вида `08-06-2001 23:59:00.000`.

In [ ]:
assert DATA_PATH.exists(), f"Не найден файл {DATA_PATH}"

df = pd.read_csv(DATA_PATH)
missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
assert not missing, f"В CSV отсутствуют обязательные колонки: {missing}"

df[TIMESTAMP] = pd.to_datetime(df[TIMESTAMP], dayfirst=True, errors="coerce")
df = df.dropna(subset=REQUIRED_COLUMNS).copy()
sort_cols = [CASE_ID, TIMESTAMP]
if "eventID" in df.columns:
    sort_cols.append("eventID")
df = df.sort_values(sort_cols).reset_index(drop=True)

for col in [CASE_ID, ACTIVITY]:
    df[col] = df[col].astype(str)

print(f"Events: {len(df):,}")
print(f"Cases: {df[CASE_ID].nunique():,}")
print(f"Activities: {df[ACTIVITY].nunique():,}")
print(f"Time range: {df[TIMESTAMP].min()} — {df[TIMESTAMP].max()}")
display(df.head())

## 2. EDA процесса

Смотрим распределение activity, число событий на case и длительность процесса. Это нужно, чтобы понимать, какие варианты процесса доминируют и насколько много длинных/аномальных cases.

In [ ]:
activity_counts = df[ACTIVITY].value_counts()
case_event_counts = df.groupby(CASE_ID).size().rename("event_count")
case_time_bounds = df.groupby(CASE_ID)[TIMESTAMP].agg(start_time="min", end_time="max")
case_duration_days = ((case_time_bounds["end_time"] - case_time_bounds["start_time"]).dt.total_seconds() / 86400).rename("duration_days")

eda_summary = pd.DataFrame({
    "metric": ["events", "cases", "activities", "median_events_per_case", "median_duration_days", "mean_duration_days"],
    "value": [
        len(df),
        df[CASE_ID].nunique(),
        df[ACTIVITY].nunique(),
        case_event_counts.median(),
        case_duration_days.median(),
        case_duration_days.mean(),
    ],
})
display(eda_summary)

top_activities = activity_counts.head(15).rename_axis("activity").reset_index(name="events")
display(top_activities)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=top_activities, y="activity", x="events", ax=axes[0], color="#2f6f9f")
axes[0].set_title("Top-15 activity по числу events")
axes[0].set_xlabel("Events")
axes[0].set_ylabel("")

sns.histplot(case_duration_days.clip(upper=case_duration_days.quantile(0.99)), bins=40, ax=axes[1], color="#9f6f2f")
axes[1].set_title("Длительность case, дни (обрезано по 99% квантилю)")
axes[1].set_xlabel("Days")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eda_activity_duration.png", dpi=160, bbox_inches="tight")
plt.show()

## 3. Подготовка event log для pm4py

Для `pm4py` приводим DataFrame к стандартным именам колонок через `pm4py.format_dataframe`. Для тяжелых операций используем фиксированную подвыборку cases, а агрегации и кластеризацию считаем по всему датасету.

In [ ]:
def sample_cases(case_index, max_cases=PM4PY_SAMPLE_CASES, random_state=RANDOM_STATE):
    case_index = pd.Index(case_index).dropna().unique()
    if len(case_index) <= max_cases:
        return case_index
    rng = np.random.default_rng(random_state)
    return pd.Index(rng.choice(case_index.to_numpy(), size=max_cases, replace=False))


def make_pm4py_dataframe(source_df):
    formatted = pm4py.format_dataframe(
        source_df[[CASE_ID, ACTIVITY, TIMESTAMP]].copy(),
        case_id=CASE_ID,
        activity_key=ACTIVITY,
        timestamp_key=TIMESTAMP,
    )
    return formatted.sort_values(["case:concept:name", "time:timestamp"]).reset_index(drop=True)

pm_case_sample = sample_cases(df[CASE_ID].unique())
pm_df_sample = make_pm4py_dataframe(df[df[CASE_ID].isin(pm_case_sample)])
event_log_sample = pm4py.convert_to_event_log(pm_df_sample)

print(f"pm4py sample cases: {len(pm_case_sample):,}")
print(f"pm4py sample events: {len(pm_df_sample):,}")

## 4. Feature engineering на уровне case

Кластеризуем не отдельные события, а cases. Для каждого case собираем длительность, число событий, число уникальных activity, стоимость, частоты activity, категориальные признаки и TF-IDF по текстовым полям.

In [ ]:
available_text_cols = [col for col in TEXT_COLUMNS if col in df.columns]
available_cat_cols = [col for col in CATEGORICAL_COLUMNS if col in df.columns]

case_base = pd.DataFrame(index=pd.Index(sorted(df[CASE_ID].unique()), name=CASE_ID))
case_base = case_base.join(case_event_counts)
case_base = case_base.join(case_duration_days)
case_base["unique_activities"] = df.groupby(CASE_ID)[ACTIVITY].nunique()

if VALUE_COL in df.columns:
    value_by_case = df.groupby(CASE_ID)[VALUE_COL].agg(value_mean="mean", value_sum="sum", value_max="max")
    case_base = case_base.join(value_by_case)
else:
    case_base[["value_mean", "value_sum", "value_max"]] = 0.0

case_base = case_base.fillna(0)

activity_features = pd.crosstab(df[CASE_ID], df[ACTIVITY])
activity_features = activity_features.div(activity_features.sum(axis=1), axis=0).fillna(0)
activity_features.columns = [f"activity_share__{c}" for c in activity_features.columns]

case_static = df.groupby(CASE_ID)[available_cat_cols].first().reindex(case_base.index).fillna("UNKNOWN").astype(str)
case_text = df.groupby(CASE_ID)[available_text_cols].first().reindex(case_base.index).fillna("").astype(str)
case_text_joined = case_text.agg(" ".join, axis=1)

numeric_features = case_base[["event_count", "duration_days", "unique_activities", "value_mean", "value_sum", "value_max"]].copy()

try:
    cat_encoder = OneHotEncoder(handle_unknown="ignore", min_frequency=0.01, sparse_output=True)
except TypeError:
    cat_encoder = OneHotEncoder(handle_unknown="ignore", min_frequency=0.01, sparse=True)

cat_matrix = cat_encoder.fit_transform(case_static)
cat_names = cat_encoder.get_feature_names_out(available_cat_cols)

text_vectorizer = TfidfVectorizer(max_features=250, min_df=5, ngram_range=(1, 2), stop_words="english")
text_matrix = text_vectorizer.fit_transform(case_text_joined)
text_names = [f"text__{term}" for term in text_vectorizer.get_feature_names_out()]

scaler = StandardScaler()
num_matrix = sparse.csr_matrix(scaler.fit_transform(numeric_features))
num_names = numeric_features.columns.to_list()

activity_matrix = sparse.csr_matrix(activity_features.reindex(case_base.index).fillna(0).to_numpy())
activity_names = activity_features.columns.to_list()

X_sparse = sparse.hstack([num_matrix, activity_matrix, cat_matrix, text_matrix]).tocsr()
feature_names = np.array(num_names + activity_names + list(cat_names) + text_names)

print(f"Case feature matrix: {X_sparse.shape[0]:,} cases x {X_sparse.shape[1]:,} features")
display(case_base.describe().T)

## 5. Кластеризация процесса

Сначала сжимаем sparse-фичи через `TruncatedSVD`, затем стандартизируем. Для `KMeans` выбираем `k` по максимальному Silhouette среди `2..8`. Дополнительно строим `DBSCAN`, чтобы получить density-based сравнение и DBCV.

In [ ]:
n_components = min(30, max(2, X_sparse.shape[1] - 1))
svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
X_reduced = svd.fit_transform(X_sparse)
X_scaled = StandardScaler().fit_transform(X_reduced)

print(f"Explained variance by SVD: {svd.explained_variance_ratio_.sum():.3f}")

rng = np.random.default_rng(RANDOM_STATE)
silhouette_idx = np.arange(len(case_base))
if len(silhouette_idx) > SILHOUETTE_SAMPLE_SIZE:
    silhouette_idx = rng.choice(silhouette_idx, SILHOUETTE_SAMPLE_SIZE, replace=False)

kmeans_results = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled[silhouette_idx], labels[silhouette_idx])
    kmeans_results.append({"k": k, "silhouette": score, "model": model, "labels": labels})

kmeans_scores = pd.DataFrame([{k: v for k, v in row.items() if k not in ["model", "labels"]} for row in kmeans_results])
best_row = max(kmeans_results, key=lambda row: row["silhouette"])
best_k = best_row["k"]
kmeans_labels = best_row["labels"]
case_base["cluster_kmeans"] = kmeans_labels

print(f"Best KMeans k={best_k}, silhouette={best_row['silhouette']:.4f}")
display(kmeans_scores)

fig, ax = plt.subplots(figsize=(7, 4))
sns.lineplot(data=kmeans_scores, x="k", y="silhouette", marker="o", ax=ax)
ax.set_title("Выбор числа кластеров KMeans по Silhouette")
ax.set_xlabel("k")
ax.set_ylabel("Silhouette")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "kmeans_silhouette_selection.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
min_samples = 25
neighbors = NearestNeighbors(n_neighbors=min_samples, metric="euclidean", n_jobs=-1)
neighbors.fit(X_scaled)
distances, _ = neighbors.kneighbors(X_scaled)
k_distance = distances[:, -1]

dbscan_candidates = []
for quantile in [0.85, 0.90, 0.95]:
    eps = float(np.quantile(k_distance, quantile))
    labels = DBSCAN(eps=eps, min_samples=min_samples, metric="euclidean", n_jobs=-1).fit_predict(X_scaled)
    cluster_count = len(set(labels)) - (1 if -1 in labels else 0)
    noise_share = float(np.mean(labels == -1))
    dbscan_candidates.append({"quantile": quantile, "eps": eps, "cluster_count": cluster_count, "noise_share": noise_share, "labels": labels})

dbscan_table = pd.DataFrame([{k: v for k, v in row.items() if k != "labels"} for row in dbscan_candidates])
valid_dbscan = [row for row in dbscan_candidates if row["cluster_count"] >= 2 and row["noise_share"] < 0.80]
chosen_dbscan = valid_dbscan[0] if valid_dbscan else dbscan_candidates[-1]
dbscan_labels = chosen_dbscan["labels"]
case_base["cluster_dbscan"] = dbscan_labels

display(dbscan_table)
print(f"Chosen DBSCAN eps={chosen_dbscan['eps']:.4f}, clusters={chosen_dbscan['cluster_count']}, noise={chosen_dbscan['noise_share']:.1%}")

## 6. Метрики качества кластеризации

- **Silhouette**: чем ближе к 1, тем лучше разделены кластеры; около 0 означает пересечение групп.
- **DBCV**: density-based метрика, полезная для DBSCAN/HDBSCAN; отрицательные значения означают слабую density-структуру.
- **Fitness**: process mining метрика, показывающая, насколько Petri net воспроизводит event log.

In [ ]:
def safe_silhouette(X, labels, sample_size=SILHOUETTE_SAMPLE_SIZE):
    labels = np.asarray(labels)
    valid = labels != -1
    if len(set(labels[valid])) < 2:
        return np.nan
    idx = np.where(valid)[0]
    if len(idx) > sample_size:
        idx = np.random.default_rng(RANDOM_STATE).choice(idx, sample_size, replace=False)
    return float(silhouette_score(X[idx], labels[idx]))


def safe_dbcv(X, labels, sample_size=DBCV_SAMPLE_SIZE):
    labels = np.asarray(labels)
    valid = labels != -1
    if len(set(labels[valid])) < 2:
        return np.nan
    idx = np.where(valid)[0]
    if len(idx) > sample_size:
        idx = np.random.default_rng(RANDOM_STATE).choice(idx, sample_size, replace=False)
    try:
        return float(validity_index(X[idx].astype(np.float64), labels[idx].astype(int), metric="euclidean"))
    except Exception as exc:
        print(f"DBCV не посчитался: {exc}")
        return np.nan

cluster_quality = pd.DataFrame([
    {"algorithm": "KMeans", "clusters": len(set(kmeans_labels)), "noise_share": 0.0, "silhouette": safe_silhouette(X_scaled, kmeans_labels), "dbcv": safe_dbcv(X_scaled, kmeans_labels)},
    {"algorithm": "DBSCAN", "clusters": len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0), "noise_share": float(np.mean(dbscan_labels == -1)), "silhouette": safe_silhouette(X_scaled, dbscan_labels), "dbcv": safe_dbcv(X_scaled, dbscan_labels)},
])
display(cluster_quality)

## 7. DFG: Directly-Follows Graph

DFG показывает, какие activity чаще всего идут непосредственно друг за другом. Ниже строится общий DFG и DFG для крупнейших KMeans-кластеров.

In [ ]:
case_clusters = case_base[["cluster_kmeans", "cluster_dbscan"]].copy()
df_with_clusters = df.merge(case_clusters, left_on=CASE_ID, right_index=True, how="left")


def compute_dfg_edges(event_df):
    edges = Counter()
    starts = Counter()
    ends = Counter()
    for trace in event_df.groupby(CASE_ID, sort=False)[ACTIVITY].agg(list):
        if not trace:
            continue
        starts[trace[0]] += 1
        ends[trace[-1]] += 1
        for a, b in zip(trace, trace[1:]):
            edges[(a, b)] += 1
    return edges, starts, ends


def plot_dfg(event_df, title, output_name, top_edges=DFG_TOP_EDGES):
    edges, starts, ends = compute_dfg_edges(event_df)
    top = edges.most_common(top_edges)
    graph = nx.DiGraph()
    for (src, dst), weight in top:
        graph.add_edge(src, dst, weight=weight)

    fig, ax = plt.subplots(figsize=(15, 10))
    if graph.number_of_edges() == 0:
        ax.text(0.5, 0.5, "Недостаточно переходов для DFG", ha="center", va="center")
        ax.axis("off")
    else:
        pos = nx.spring_layout(graph, seed=RANDOM_STATE, k=1.4)
        node_sizes = [700 + 80 * graph.degree(node) for node in graph.nodes]
        max_weight = max([data["weight"] for _, _, data in graph.edges(data=True)])
        edge_widths = [1 + 4 * data["weight"] / max_weight for _, _, data in graph.edges(data=True)]
        nx.draw_networkx_nodes(graph, pos, node_size=node_sizes, node_color="#d6e8f6", edgecolors="#315f7d", ax=ax)
        nx.draw_networkx_edges(graph, pos, width=edge_widths, arrows=True, arrowstyle="-|>", arrowsize=15, edge_color="#777777", ax=ax)
        nx.draw_networkx_labels(graph, pos, font_size=8, ax=ax)
        edge_labels = {(u, v): graph[u][v]["weight"] for u, v in graph.edges}
        nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels, font_size=7, ax=ax)
        ax.axis("off")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / output_name, dpi=160, bbox_inches="tight")
    plt.show()

plot_dfg(df_with_clusters, "DFG всего процесса: top directly-follow edges", "dfg_full_process.png")

largest_clusters = case_base["cluster_kmeans"].value_counts().head(3).index.tolist()
for cluster_id in largest_clusters:
    cluster_events = df_with_clusters[df_with_clusters["cluster_kmeans"] == cluster_id]
    plot_dfg(cluster_events, f"DFG KMeans-кластера {cluster_id}: top directly-follow edges", f"dfg_kmeans_cluster_{cluster_id}.png", top_edges=35)

## 8. Сеть Петри и Fitness через Alpha miner

По условию строим Petri net через Alpha miner из `pm4py`. Fitness считаем token replay на той же фиксированной подвыборке cases, чтобы шаг был воспроизводимым и не слишком тяжелым.

In [ ]:
net, initial_marking, final_marking = alpha_miner.apply(event_log_sample)
fitness_result = replay_fitness_evaluator.apply(
    event_log_sample,
    net,
    initial_marking,
    final_marking,
    variant=replay_fitness_evaluator.Variants.TOKEN_BASED,
)

display(pd.DataFrame([fitness_result]))

try:
    petri_gviz = pn_visualizer.apply(net, initial_marking, final_marking)
    pn_visualizer.save(petri_gviz, str(OUTPUT_DIR / "petri_net_alpha.png"))
    display(petri_gviz)
except Exception as exc:
    print("Petri net построена, но визуализация не сохранилась. Частая причина — не установлен системный Graphviz.")
    print(exc)

## 9. Process metrics по кластерам

Считаем размер кластера, среднюю/медианную длительность, число событий и типичные activity. Это помогает интерпретировать, что именно отделила кластеризация.

In [ ]:
cluster_summary = (
    case_base.groupby("cluster_kmeans")
    .agg(
        cases=("event_count", "size"),
        avg_events=("event_count", "mean"),
        median_events=("event_count", "median"),
        avg_duration_days=("duration_days", "mean"),
        median_duration_days=("duration_days", "median"),
        avg_value=("value_mean", "mean"),
    )
    .sort_values("cases", ascending=False)
)
cluster_summary["case_share"] = cluster_summary["cases"] / cluster_summary["cases"].sum()
display(cluster_summary)

cluster_activity_rows = []
for cluster_id in cluster_summary.index:
    sub = df_with_clusters[df_with_clusters["cluster_kmeans"] == cluster_id]
    top = sub[ACTIVITY].value_counts(normalize=True).head(5)
    cluster_activity_rows.append({"cluster_kmeans": cluster_id, "top_activities": "; ".join([f"{idx} ({val:.1%})" for idx, val in top.items()])})
cluster_activity_summary = pd.DataFrame(cluster_activity_rows)
display(cluster_activity_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cluster_summary.reset_index().plot.bar(x="cluster_kmeans", y="cases", ax=axes[0], legend=False, color="#427aa1")
axes[0].set_title("Размер KMeans-кластеров")
axes[0].set_xlabel("Cluster")
axes[0].set_ylabel("Cases")

cluster_summary.reset_index().plot.bar(x="cluster_kmeans", y="avg_duration_days", ax=axes[1], legend=False, color="#a16f42")
axes[1].set_title("Средняя длительность case по кластерам")
axes[1].set_xlabel("Cluster")
axes[1].set_ylabel("Days")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cluster_process_metrics.png", dpi=160, bbox_inches="tight")
plt.show()

## 10. Самые частые циклы

Ищем два типа повторов: повторяющиеся activity внутри trace и возвраты вида `A -> ... -> A`, где activity появляется повторно после одного или нескольких других событий.

In [ ]:
traces = df.groupby(CASE_ID, sort=False)[ACTIVITY].agg(list)

repeated_activity_counter = Counter()
return_pattern_counter = Counter()

for trace in traces:
    counts = Counter(trace)
    for activity, count in counts.items():
        if count > 1:
            repeated_activity_counter[activity] += count - 1

    last_position = {}
    for idx, activity in enumerate(trace):
        if activity in last_position and idx - last_position[activity] > 1:
            middle = trace[last_position[activity] + 1: idx]
            if middle:
                return_pattern_counter[f"{activity} -> ... -> {activity}"] += 1
        last_position[activity] = idx

repeated_activity_df = pd.DataFrame(repeated_activity_counter.most_common(15), columns=["activity", "extra_repetitions"])
return_patterns_df = pd.DataFrame(return_pattern_counter.most_common(15), columns=["cycle_pattern", "count"])

display(repeated_activity_df)
display(return_patterns_df)

most_frequent_cycle = return_patterns_df.iloc[0].to_dict() if len(return_patterns_df) else {"cycle_pattern": "not found", "count": 0}
print("Самый частый цикл:", most_frequent_cycle)

## 11. Feature importance: что влияет на длительность процесса

Target: `duration_days`. Модель: `RandomForestRegressor`. Чтобы не допустить утечки, из признаков для модели исключаем сам `duration_days`, но оставляем activity, категориальные и текстовые признаки.

In [ ]:
target = case_base["duration_days"].clip(lower=0)
model_numeric_features = case_base[["event_count", "unique_activities", "value_mean", "value_sum", "value_max"]].copy()
model_num_matrix = sparse.csr_matrix(StandardScaler().fit_transform(model_numeric_features))
model_num_names = model_numeric_features.columns.to_list()

X_duration = sparse.hstack([model_num_matrix, activity_matrix, cat_matrix, text_matrix]).tocsr()
duration_feature_names = np.array(model_num_names + activity_names + list(cat_names) + text_names)

X_train, X_test, y_train, y_test = train_test_split(X_duration, target, test_size=0.20, random_state=RANDOM_STATE)

rf = RandomForestRegressor(n_estimators=180, max_depth=12, min_samples_leaf=10, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
train_r2 = rf.score(X_train, y_train)
test_r2 = rf.score(X_test, y_test)

importance_df = pd.DataFrame({"feature": duration_feature_names, "importance": rf.feature_importances_}).sort_values("importance", ascending=False).head(25)

print(f"RandomForest R^2 train={train_r2:.3f}, test={test_r2:.3f}")
display(importance_df)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=importance_df.head(15), y="feature", x="importance", ax=ax, color="#4f8f6f")
ax.set_title("Top feature importance для длительности процесса")
ax.set_xlabel("Importance")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "duration_feature_importance.png", dpi=160, bbox_inches="tight")
plt.show()

## 12. Кластеризация и суммаризация текста

Для текстового блока используем TF-IDF по case-level тексту. Суммаризация делается через top TF-IDF terms и типичные примеры значений из исходных текстовых колонок.

In [ ]:
text_cluster_vectorizer = TfidfVectorizer(max_features=500, min_df=3, ngram_range=(1, 2), stop_words="english")
X_text_cluster = text_cluster_vectorizer.fit_transform(case_text_joined)
text_terms = np.array(text_cluster_vectorizer.get_feature_names_out())

text_k = min(6, max(2, int(np.sqrt(len(case_text_joined) / 2_000)) + 2))
text_kmeans = KMeans(n_clusters=text_k, n_init=20, random_state=RANDOM_STATE)
text_labels = text_kmeans.fit_predict(X_text_cluster)
case_base["cluster_text"] = text_labels

text_silhouette_idx = np.arange(X_text_cluster.shape[0])
if len(text_silhouette_idx) > SILHOUETTE_SAMPLE_SIZE:
    text_silhouette_idx = np.random.default_rng(RANDOM_STATE).choice(text_silhouette_idx, SILHOUETTE_SAMPLE_SIZE, replace=False)
text_silhouette = silhouette_score(X_text_cluster[text_silhouette_idx], text_labels[text_silhouette_idx])
print(f"Text clusters: {text_k}, silhouette={text_silhouette:.4f}")

text_summary_rows = []
centers = text_kmeans.cluster_centers_
for cluster_id in range(text_k):
    idx = np.where(text_labels == cluster_id)[0]
    top_terms = text_terms[np.argsort(centers[cluster_id])[-12:][::-1]]
    cases_in_cluster = case_base.index[idx]
    examples = case_text.loc[cases_in_cluster].head(5)
    example_values = []
    for col in available_text_cols:
        vals = examples[col].replace("", np.nan).dropna().unique()[:3]
        if len(vals):
            example_values.append(f"{col}: {', '.join(map(str, vals))}")
    text_summary_rows.append({"cluster_text": cluster_id, "cases": len(idx), "case_share": len(idx) / len(case_base), "top_terms": ", ".join(top_terms), "examples": " | ".join(example_values)})

text_summary = pd.DataFrame(text_summary_rows).sort_values("cases", ascending=False)
display(text_summary)

## 13. Автоматические выводы для отчета

Ниже формируется краткий текст, который можно использовать как основу защиты. Он опирается на реальные метрики, рассчитанные выше.

In [ ]:
largest_cluster_id = int(cluster_summary.index[0])
slowest_cluster_id = int(cluster_summary["avg_duration_days"].idxmax())
fastest_cluster_id = int(cluster_summary["avg_duration_days"].idxmin())

best_quality = cluster_quality.loc[cluster_quality["algorithm"] == "KMeans"].iloc[0]
fitness_value = fitness_result.get("log_fitness", fitness_result.get("average_trace_fitness", np.nan))

conclusion = f"""
### Итоговые выводы

1. В датасете {len(df):,} events, {df[CASE_ID].nunique():,} cases и {df[ACTIVITY].nunique():,} уникальных activity. Процесс неоднородный: cases различаются по длительности, числу событий, типам закупочных документов, поставщикам и spend area.

2. Для кластеризации cases использовались процессные признаки, activity shares, категориальные атрибуты и TF-IDF по текстовым полям. Лучший KMeans выбран с k={best_k}; Silhouette={best_quality['silhouette']:.3f}. Если значение не очень высокое, это означает, что процессные варианты пересекаются, а не образуют идеально отделимые группы.

3. Самый крупный KMeans-кластер: {largest_cluster_id}. Самый медленный по средней длительности: {slowest_cluster_id}. Самый быстрый: {fastest_cluster_id}. Это удобно использовать для бизнес-интерпретации: сравнить top activity и атрибуты этих кластеров.

4. Fitness Alpha/Petri модели на подвыборке: {fitness_value:.3f}. Если fitness ниже 1, значит Alpha miner не полностью воспроизводит все реальные варианты процесса. Для BPI-подобных закупочных логов это ожидаемо: процесс содержит исключения, возвраты, отмены и редкие ветки.

5. Самый частый цикл: `{most_frequent_cycle['cycle_pattern']}` с числом повторов {most_frequent_cycle['count']}. Частые циклы указывают на rework: изменения заказа, отмены, повторные invoice/goods receipt или возвраты к уже выполненным activity.

6. Feature importance показывает, какие признаки сильнее всего связаны с длительностью процесса. В отчете стоит отдельно прокомментировать top-10 факторов: это могут быть число событий, уникальные activity, конкретные типы activity, spend area, vendor/company или текстовые категории.

7. Текстовая кластеризация группирует cases по spend/name/classification описаниям. Top terms по каждому текстовому кластеру дают короткую интерпретацию предметной области закупки.
"""

display(Markdown(conclusion))

## 14. Что говорить на защите

- **Подход**: я перевел event log в case-level признаки, потому что кластеризовать нужно варианты процесса, а не отдельные events.
- **Кластеры**: KMeans выбран по Silhouette; DBSCAN использован как density-based baseline и для DBCV.
- **Метрики**: Silhouette/DBCV оценивают геометрию кластеров, Fitness оценивает соответствие process model реальному логу.
- **DFG и Petri net**: DFG показывает частые прямые переходы; Petri net показывает формальную модель процесса.
- **Почему качество может быть умеренным**: закупочный процесс имеет много вариантов, редких веток, отмен, изменений и повторных событий, поэтому кластеры не обязаны быть идеально отделимыми.
- **Бизнес-вывод**: самые долгие и цикличные кластеры стоит проверять на rework, ручные корректировки, отмены и проблемные типы документов/поставщиков.